In [ ]:
import scipy.io as sio
import numpy as np
import torch.nn as nn
import yaml
import torch
import argparse
from tqdm import tqdm
from torch.autograd import Variable
import torch.optim as optim
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
%matplotlib inline

# 设置中文字体支持
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans', 'Arial Unicode MS', 'WenQuanYi Micro Hei']
matplotlib.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

# define device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
import os
# 移除 expandable_segments 选项，因为它在较旧的 PyTorch 版本中不被支持
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True)
set_seed(42)

In [ ]:
with open(r'ns.yaml', 'r') as stream:                           
    config = yaml.load(stream, yaml.FullLoader)
    print(config)


In [ ]:
import struct
from pathlib import Path
from typing import List, Tuple, Union
import numpy as np


def read_mat_frames_simple(mat_path: Union[str, Path]) -> List[Tuple[str, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]]:
    """
    读取 FluidX3D 导出的 MAT(v4) 文件，不做网格重排。
    返回列表，每个元素是 (name, x, y, t, u, v, p, mask_bcxy, mask_load)，均为 1D 向量。
    列位约定（0-based 下标）：
      1->x, 2->y, 4->u, 5->v, 7->p, 8->mask_bcxy, 9->mask_load, 10->t
    要求每帧至少 11 列。
    """
    path = Path(mat_path)
    frames: List[Tuple[str, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]] = []

    with path.open("rb") as f:
        while True:
            header = f.read(20)
            if len(header) == 0:
                break
            if len(header) < 20:
                raise RuntimeError("MAT 头不完整。")

            var_type, mrows, ncols, imagf, namelen = struct.unpack("<5i", header)
            name_bytes = f.read(namelen)
            if len(name_bytes) != namelen:
                raise RuntimeError("读取变量名时意外结束。")
            # 名字块按 8 字节对齐
            if namelen % 8 != 0:
                f.seek(8 - (namelen % 8), 1)
            name = name_bytes.rstrip(b"\x00").decode("ascii", errors="ignore")

            data_count = int(mrows) * int(ncols)
            if data_count < 0:
                raise RuntimeError(f"变量 {name} 数据量为负。")
            data = np.fromfile(f, dtype=np.float64, count=data_count)
            if imagf != 0:
                raise NotImplementedError("不支持复数数据。")
            if data.size != data_count:
                raise RuntimeError(f"读取变量 {name} 数据时意外结束。")

            # MAT v4 为 Fortran 列主序
            data = data.reshape((mrows, ncols), order="F")

            if name.startswith("cell_t"):
                if ncols < 11:
                    raise ValueError(f"{name} 的列数为 {ncols}，但需要至少 11 列。")
                # 取列（保持为 1D）

                x = data[:, 1].copy()
                y = data[:, 2].copy()
                u = data[:, 4].copy()
                v = data[:, 5].copy()
                p = data[:, 7].copy()
                mask_bcxy = data[:, 8].copy()
                mask_load = data[:, 9].copy()
                t = data[:, 10].copy()

                frames.append((name, x, y, t, u, v, p, mask_bcxy, mask_load))
            else:
                # 跳过其他变量（如 time_steps、column_labels 等）
                pass

    # 依据名字中的时间步编号排序，确保顺序正确
    frames.sort(key=lambda item: int(item[0].split("t")[1]))
    return frames


def convert_frames_to_arrays(
    frames_data: List[Union[
        Tuple[str, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray],  # (name, x, y, t, u, v, p, mask_bcxy, mask_load)
        Tuple[str, np.ndarray],  # (name, frame2d)
        np.ndarray  # frame2d
    ]]
):
    """
    将 frames_data 统一转换为：
      up, vp, pp, coor1, flag_BCxy1, flag_load1, times, flag_ID
    - up/vp/pp: List[np.ndarray]，每帧 1D
    - coor1: List[np.ndarray]，每帧 (N,2)
    - flag_BCxy1/flag_load1: List[np.ndarray]，float32 1D
    - times: List[np.ndarray]（每点时间 1D）；若要帧级标量时间，可在外部改为 float(t[0])
    - flag_ID: np.ndarray，帧编号 [1..num_frames]
    兼容三种输入：
      1) (name, x, y, t, u, v, p, mask_bcxy, mask_load)
      2) (name, frame2d) 其中 frame2d.shape = (N, >=11)
      3) frame2d 直接就是二维数组
    """
    def to_arrays(item):
        if isinstance(item, tuple) and len(item) == 9 and isinstance(item[1], np.ndarray):
            # (name, x, y, t, u, v, p, mask_bcxy, mask_load)
            _, x, y, t, u, v, p, mask_bcxy, mask_load = item
            return x, y, t, u, v, p, mask_bcxy, mask_load
        if isinstance(item, tuple) and len(item) == 2 and isinstance(item[1], np.ndarray):
            # (name, frame2d)
            _, f = item
            if f.ndim != 2 or f.shape[1] < 11:
                raise ValueError(f"二维帧形状异常: {f.shape}, 需要至少 11 列")
            return f[:,1], f[:,2], f[:,10], f[:,4], f[:,5], f[:,7], f[:,8], f[:,9]
        # 直接就是 frame2d
        f = item
        if not isinstance(f, np.ndarray) or f.ndim != 2 or f.shape[1] < 11:
            raise ValueError(f"帧形状/类型异常: {type(f)} {getattr(f, 'shape', None)}, 需要二维数组且至少 11 列")
        return f[:,1], f[:,2], f[:,10], f[:,4], f[:,5], f[:,7], f[:,8], f[:,9]

    up: List[np.ndarray] = []
    vp: List[np.ndarray] = []
    pp: List[np.ndarray] = []
    coor1: List[np.ndarray] = []
    flag_BCxy1: List[np.ndarray] = []
    flag_load1: List[np.ndarray] = []
    times: List[np.ndarray] = []

    for item in frames_data:
        x, y, t, u, v, p, mask_bcxy, mask_load = to_arrays(item)
        coords = np.column_stack([x, y])

        flag_BCxy = mask_bcxy.astype(np.float32)
        flag_load = (mask_load > 0).astype(np.float32)

        up.append(u)
        vp.append(v)
        pp.append(p)
        coor1.append(coords)
        flag_BCxy1.append(flag_BCxy)
        flag_load1.append(flag_load)
        times.append(t)

    num_frames = len(frames_data)
    flag_ID = np.arange(1, num_frames + 1)

    # 返回 numpy 对象数组，方便不同帧节点数不同的情况
    return (
        np.array(up, dtype=object),
        np.array(vp, dtype=object),
        np.array(pp, dtype=object),
        np.array(coor1, dtype=object),
        np.array(flag_BCxy1, dtype=object),
        np.array(flag_load1, dtype=object),
        np.array(times, dtype=object),
        flag_ID
    )


def downsample_data(
    up, vp, pp, coor1, flag_BCxy1, flag_load1, times, flag_ID,
    spatial_ratio=1.0, 
    temporal_ratio=1.0,
    preserve_boundary=True,
    random_seed=42
):
    """
    对数据进行下采样
    
    参数:
        spatial_ratio: 空间下采样比例 (0-1)，例如 0.5 表示保留50%的点
        temporal_ratio: 时间下采样比例 (0-1)，例如 0.5 表示保留50%的帧
        preserve_boundary: 是否保留边界点（flag_BCxy==1 或 flag_load==1 的点）
        random_seed: 随机种子，确保可重复性
    
    返回:
        下采样后的数据，格式与输入相同
    """
    np.random.seed(random_seed)
    
    num_frames = len(up)
    
    # 时间下采样：选择要保留的帧
    if temporal_ratio < 1.0:
        num_frames_to_keep = int(num_frames * temporal_ratio)
        frame_indices = np.sort(np.random.choice(num_frames, num_frames_to_keep, replace=False))
    else:
        frame_indices = np.arange(num_frames)
    
    up_downsampled = []
    vp_downsampled = []
    pp_downsampled = []
    coor1_downsampled = []
    flag_BCxy1_downsampled = []
    flag_load1_downsampled = []
    times_downsampled = []
    flag_ID_downsampled = []
    
    for i in frame_indices:
        # 获取当前帧的数据
        u_frame = up[i]
        v_frame = vp[i]
        p_frame = pp[i]
        coor_frame = coor1[i]
        flag_BCxy_frame = flag_BCxy1[i]
        flag_load_frame = flag_load1[i]
        time_frame = times[i]
        
        num_points = len(u_frame)
        
        # 空间下采样
        if spatial_ratio < 1.0:
            num_points_to_keep = int(num_points * spatial_ratio)
            
            if preserve_boundary:
                # 分离边界点和内部点
                boundary_mask = (flag_BCxy_frame == 1) | (flag_load_frame == 1)
                interior_mask = ~boundary_mask
                
                boundary_indices = np.where(boundary_mask)[0]
                interior_indices = np.where(interior_mask)[0]
                
                # 保留所有边界点
                num_interior_to_keep = max(0, num_points_to_keep - len(boundary_indices))
                
                if num_interior_to_keep > 0 and len(interior_indices) > 0:
                    # 从内部点中随机采样
                    if num_interior_to_keep < len(interior_indices):
                        sampled_interior = np.random.choice(
                            interior_indices, 
                            size=num_interior_to_keep, 
                            replace=False
                        )
                    else:
                        sampled_interior = interior_indices
                    
                    # 合并边界点和采样的内部点
                    selected_indices = np.sort(np.concatenate([boundary_indices, sampled_interior]))
                else:
                    # 如果边界点已经足够多，只保留边界点
                    selected_indices = boundary_indices
            else:
                # 不保留边界点，随机采样
                selected_indices = np.sort(
                    np.random.choice(num_points, num_points_to_keep, replace=False)
                )
            
            # 应用下采样
            up_downsampled.append(u_frame[selected_indices])
            vp_downsampled.append(v_frame[selected_indices])
            pp_downsampled.append(p_frame[selected_indices])
            coor1_downsampled.append(coor_frame[selected_indices])
            flag_BCxy1_downsampled.append(flag_BCxy_frame[selected_indices])
            flag_load1_downsampled.append(flag_load_frame[selected_indices])
            times_downsampled.append(time_frame[selected_indices])
        else:
            # 不进行空间下采样
            up_downsampled.append(u_frame)
            vp_downsampled.append(v_frame)
            pp_downsampled.append(p_frame)
            coor1_downsampled.append(coor_frame)
            flag_BCxy1_downsampled.append(flag_BCxy_frame)
            flag_load1_downsampled.append(flag_load_frame)
            times_downsampled.append(time_frame)
        
        flag_ID_downsampled.append(flag_ID[i])
    
    # 更新 flag_ID 为连续编号
    flag_ID_downsampled = np.arange(1, len(flag_ID_downsampled) + 1)
    
    return (
        np.array(up_downsampled, dtype=object),
        np.array(vp_downsampled, dtype=object),
        np.array(pp_downsampled, dtype=object),
        np.array(coor1_downsampled, dtype=object),
        np.array(flag_BCxy1_downsampled, dtype=object),
        np.array(flag_load1_downsampled, dtype=object),
        np.array(times_downsampled, dtype=object),
        flag_ID_downsampled
    )


if __name__ == "__main__":
    # 1) 读取多个 MAT 文件
    mat_files = [
        Path("/root/shuntai/data_cells_u50.mat"),
        Path("/root/shuntai/data_cells_u51.mat"),
        Path("/root/shuntai/data_cells_u52.mat"),
        Path("/root/shuntai/data_cells_u53.mat"),
        Path("/root/shuntai/data_cells_u54.mat"),
    ]
    frames: List[Tuple[str, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]] = []
    for mat_path in mat_files:
        if not mat_path.exists():
            raise FileNotFoundError(f"未找到数据文件: {mat_path}")
        print(f"正在加载数据文件: {mat_path}")
        file_frames = read_mat_frames_simple(mat_path)
        print(f"  {mat_path.name} -> {len(file_frames)} 个时间步")
        frames.extend(file_frames)
    print(f"共合并 {len(frames)} 个时间步。")

    # 2) 转换到期望的数据结构
    up, vp, pp, coor1, flag_BCxy1, flag_load1, times, flag_ID = convert_frames_to_arrays(frames)

    # 2.5) 对数据进行下采样
    # 可以调整以下参数：
    # - spatial_ratio: 空间下采样比例 (0-1)，例如 0.5 表示保留50%的点
    # - temporal_ratio: 时间下采样比例 (0-1)，例如 0.5 表示保留50%的帧
    # - preserve_boundary: 是否保留边界点（建议设为True）
    print("正在进行下采样...")
    up, vp, pp, coor1, flag_BCxy1, flag_load1, times, flag_ID = downsample_data(
        up, vp, pp, coor1, flag_BCxy1, flag_load1, times, flag_ID,
        spatial_ratio=0.8,      # 保留50%的空间点
        temporal_ratio=1.0,      # 保留100%的时间帧（不进行时间下采样）
        preserve_boundary=True,  # 保留所有边界点
        random_seed=42
    )
    print("下采样完成！")

    # 3) 打印形状检查
    print('原始数据形状检查:')
    print(f'  up: {up.shape}, 每个元素形状: {[up[i].shape for i in range(min(3, len(up)))]}')
    print(f'  vp: {vp.shape}, 每个元素形状: {[vp[i].shape for i in range(min(3, len(vp)))]}')
    print(f'  pp: {pp.shape}, 每个元素形状: {[pp[i].shape for i in range(min(3, len(pp)))]}')
    print(f'  coor1: {coor1.shape}, 每个元素形状: {[coor1[i].shape for i in range(min(3, len(coor1)))]}')
    print(f'  flag_BCxy1: {flag_BCxy1.shape}')
    print(f'  flag_load1: {flag_load1.shape}')
    print(f'  times: {times.shape}')
    print(f'  flag_ID: {flag_ID.shape}  范围: [{flag_ID[0]}..{flag_ID[-1]}]')
    print(f'  times: {times}')
    # 4) 若需要“每帧一个标量时间”，可在此派生：
    frame_time_scalar = np.array([float(t[0]) if isinstance(t, np.ndarray) and t.size > 0 else np.nan for t in times])
    print(f'  帧级时间标量(前3): {frame_time_scalar[:10]}')

In [ ]:
# 获取 u, v, p 的元素数量
datasize = len(up)
max_pde_nodes = 0
max_par_nodes = 0
max_bcxy_nodes = 0
for i in range(datasize):
    num_pde = np.sum((1-flag_load1[i])*(1-flag_BCxy1[i]))
    # num_pde = np.sum((flag_bc[i]))  
    if num_pde > max_pde_nodes:
        max_pde_nodes = num_pde 

    num_par_ = np.sum((flag_load1[i]))   
    if num_par_ > max_par_nodes:
        max_par_nodes = num_par_

    num_bcxy = np.sum(flag_BCxy1[i])
    if num_bcxy > max_bcxy_nodes:
        max_bcxy_nodes = num_bcxy
max_pde_nodes = int(max_pde_nodes)
max_bcxy_nodes = int(max_bcxy_nodes)
max_par_nodes = int(max_par_nodes)

# 打印最大节点数量
print(f"datasize: {datasize}")
print(f"PDE nodes: {max_pde_nodes}")
print(f"Load nodes: {max_par_nodes}")
print(f"BCxy nodes: {max_bcxy_nodes}")
print(f"all nodes: {max_pde_nodes + max_par_nodes  +max_bcxy_nodes}")

In [ ]:
u_list = []
v_list = []
p_list = []

for i in range(datasize):
    # 填充 u, v, p
    u1 = up[i].reshape(-1)
    v1 = vp[i].reshape(-1)
    p1 = pp[i].reshape(-1)
    # 将每次循环的结果添加到列表
    u_list.append(u1)
    v_list.append(v1)
    p_list.append(p1)

# 最后将所有结果合并为一个大的数组
u_combined = np.concatenate(u_list, axis=0)
v_combined = np.concatenate(v_list, axis=0)
p_combined = np.concatenate(p_list, axis=0)

# 分别计算 u_combined, v_combined, p_combined 的最小值和最大值
min_u = np.min(u_combined)
max_u = np.max(u_combined)

min_v = np.min(v_combined)
max_v = np.max(v_combined)

min_p = np.min(p_combined)
max_p = np.max(p_combined)

# 输出结果
print(f"u_combined - min: {min_u}, max: {max_u}")
print(f"v_combined - min: {min_v}, max: {max_v}")
print(f"p_combined - min: {min_p}, max: {max_p}")


In [ ]:
# 获取每个数组的实际形状
up_shape = [len(up[i]) for i in range(up.shape[0])]
vp_shape = [len(vp[i]) for i in range(vp.shape[0])]
pp_shape = [len(pp[i]) for i in range(pp.shape[0])]

# 获取 coor1 的形状：每个子数组是 (N, 2)
coor_shape = [len(coor1[i]) for i in range(coor1.shape[0])]  # 每个子数组的长度 N

flag_BCxy_shape = [len(flag_BCxy1[i]) for i in range(flag_BCxy1.shape[0])]
flag_load_shape = [len(flag_load1[i]) for i in range(flag_load1.shape[0])]

# 初始化 u, v, p, flag_BCxy, flag_BCy, flag_load
u = np.empty((up.shape[0], max(up_shape)))  # 初始化 u，取最长的列
v = np.empty((vp.shape[0], max(vp_shape)))  # 初始化 v，取最长的列
p = np.empty((pp.shape[0], max(pp_shape)))  # 初始化 p，取最长的列
# 初始化 coor 为 (100, 最大长度, 2)
coor = np.empty((coor1.shape[0], max(coor_shape), 2))  # 最后一维是 2

# 初始化 flag_BCxy, flag_BCy, flag_load，取最长的列
flag_BCxy = np.empty((flag_BCxy1.shape[0], max(flag_BCxy_shape)))
flag_load = np.empty((flag_load1.shape[0], max(flag_load_shape)))
Times = np.empty((coor1.shape[0], max(coor_shape)))

# 按行填充 up, vp, pp 和 flag_BCxy, flag_BCy, flag_load

for i in range(datasize):

    # 使用逐点时间（来自第 10 列）
    t_array = times[i].reshape(-1)
    up_ = up[i].reshape(-1) 
    vp_ = vp[i].reshape(-1)  
    pp_ = pp[i].reshape(-1) 
    
    # 使用各自的最小值和最大值进行最小-最大归一化
    up_ = (up_ - min_u) / (max_u - min_u)
    vp_ = (vp_ - min_v) / (max_v - min_v)
    pp_ = (pp_ - min_p) / (max_p - min_p)

    # 填充 u, v, p
    u[i, :up_shape[i]] = up_
    v[i, :vp_shape[i]] = vp_
    p[i, :pp_shape[i]] = pp_
    
    # 填充 coor1 到 coor
    coor[i, :coor_shape[i], :] = coor1[i]  # 保持 coor1[i] 的二维结构，填充到 coor
    Times[i, :coor_shape[i]] = t_array

    
    # 填充 flag_BCxy, flag_BCy, flag_load
    flag_BCxy[i, :flag_BCxy_shape[i]] = flag_BCxy1[i].reshape(-1)
    flag_load[i, :flag_load_shape[i]] = flag_load1[i].reshape(-1)

    # 将未填充的部分设为零
    u[i, up_shape[i]:] = 0
    v[i, vp_shape[i]:] = 0
    p[i, pp_shape[i]:] = 0
    coor[i, coor_shape[i]:, :] = 0 
    flag_BCxy[i, flag_BCxy_shape[i]:] = 0
    flag_load[i, flag_load_shape[i]:] = 0
    Times[i, coor_shape[i]:] = 0

print(u.shape, v.shape, p.shape)
print(flag_BCxy.shape, flag_load.shape)
print(coor.shape)
print(Times.shape)

In [ ]:
nn=1
u_50 = u[nn, :]
v_50 = v[nn, :]
p_50 = p[nn, :]
flag_BCxy_50 = flag_BCxy[nn, :]
flag_load_50 = flag_load[nn, :]
coor_50 = coor[nn, :, :]  # 形状为 (40000, 2)

# 筛选出不同边界条件的坐标
mask_BCxy = (flag_BCxy_50 == 1)  # BCxy边界条件
mask_outlet = (flag_load_50 == 1)  # load边界条件
valid_mask_50 = (flag_BCxy_50 ==0)  & (flag_load_50 == 0)  # 有效内部点

# 获取不同区域的坐标
coors_BCxy = coor_50[mask_BCxy]# coors_BCy = coor_50[mask_BCy]
coors_load = coor_50[mask_outlet]
valid_coors_50 = coor_50[valid_mask_50]

# 获取有效坐标的速度
valid_u_50 = u_50[valid_mask_50]
valid_v_50 = v_50[valid_mask_50]
speed_50 = np.sqrt(valid_u_50**2 + valid_v_50**2)

# 创建一个图形窗口，设置长宽比为 4:1
plt.figure(figsize=(16, 4))

# 绘制内部点的速度场（背景）
scatter_valid = plt.scatter(valid_coors_50[:, 0], valid_coors_50[:, 1], c=speed_50, cmap='jet', s=5, alpha=0.7, label='Interior points')

# 绘制flag_BCxy==1的区域（黑色）
if len(coors_BCxy) > 0:
    plt.scatter(coors_BCxy[:, 0], coors_BCxy[:, 1], c='black', s=10, alpha=0.8, marker='s', label='flag_BCxy==1', edgecolors='black', linewidths=0.5)

# 绘制flag_load==1的区域（蓝色）
if len(coors_load) > 0:
    plt.scatter(coors_load[:, 0], coors_load[:, 1], c='blue', s=10, alpha=0.8, marker='o', label='flag_load==1', edgecolors='darkblue', linewidths=0.5)

# 添加颜色条以表示速度的大小
plt.colorbar(scatter_valid, label='Speed')

# 设置坐标轴标签
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')

# 添加标题和图例
plt.title('Velocity Field and Boundary Conditions (Sample {})'.format(nn))
plt.legend(loc='upper right', fontsize=8)

# 显示图像
plt.show()


In [ ]:
# 假设 flag_load 和 flag_BCxy、flag_BCy 已经初始化，形状为 (100, 19815)
monitor1 = np.zeros((flag_load.shape[0], flag_load.shape[1]), dtype=int)
num_one = 0  # 每行设置 1000 个为 1 的索引
# 遍历每一行
for i in range(flag_load.shape[0]):
    # 找到 flag_BCxy, flag_BCy, flag_load 中为 0 的位置
    flag_load_indices = np.where((flag_BCxy[i] == 0) & (flag_load[i] == 0))[0]

    # 计算 num_ones 需要设置为 1 的数量
    num_bcxy = np.sum(flag_BCxy[i] == 1)
    num_ones = max_bcxy_nodes - num_bcxy + num_one

    # 确保 num_ones 不超过 flag_load_indices 的长度
    num_ones = min(num_ones, len(flag_load_indices))  # 限制 num_ones

    # 从 flag_load == 0 的位置中随机选择 num_ones 个索引
    indices = np.random.choice(flag_load_indices, size=num_ones, replace=False)

    # 将这些位置的值设置为 1
    monitor1[i, indices] = 1

# 创建 flag_bc 的副本，形状为 (100, 19815)
flag_bc = np.copy(flag_BCxy)

# 用循环逐行处理每个标志数组
for i in range(flag_BCxy.shape[0]):  # 遍历每一行
    # 判断 monitor1 是否为 1，并且 flag_load 或 flag_BCy 是否为 1
    flag_bc[i, :] = (monitor1[i, :] == 1) | (flag_load[i, :] == 1) | (flag_BCxy[i, :] == 1)

# # 打印结果：查看 monitor1 和 flag_bc 的形状
# print(monitor1.shape)
# print(flag_bc.shape)
# # 输出每组中 flag_bc == 1 的数量
# flag_bc_count = np.sum(flag_bc == 1, axis=1)  # 统计每行中 flag_bc == 1 的数量
# for i in range(len(flag_bc_count)):
#     print(f'Number of points where flag_bc == 1 in row {i}: {flag_bc_count[i]}')

In [ ]:

# 获取每行中 flag_bc == 1 的索引
id_param = []
for i in range(flag_bc.shape[0]):  # 遍历每一行
    row_indices = np.where(flag_bc[i] == 1)[0]  # 获取该行中 flag_bc == 1 的索引
    id_param.append(row_indices)  # 将每行的索引添加到列表中

# 将列表转换为二维数组，形状为 (100, 最大数量)
id_param_array = np.zeros((flag_bc.shape[0], max(len(row) for row in id_param)), dtype=int)  # 初始化一个全为0的数组

# # 将每行的索引填充到 id_param_array 中
# for i in range(flag_bc.shape[0]):
#     id_param_array[i, :len(id_param[i])] = id_param[i]  # 填充每行的索引

# # 输出结果查看
# print(id_param_array.shape)  # 应该输出 (100, N)，其中 N 是每行 flag_bc == 1 的索引数量
# print(id_param_array)


In [ ]:
# 创建一个新的数组，通过 id_param 按行选择 u, v, p 和 coor 中的元素
u_selected = np.array([u[i, id_param[i]] for i in range(u.shape[0])])  # 形状为 (100, N)
v_selected = np.array([v[i, id_param[i]] for i in range(v.shape[0])])  # 形状为 (100, N)
p_selected = np.array([p[i, id_param[i]] for i in range(p.shape[0])])  # 形状为 (100, N)
print(u_selected.shape, v_selected.shape, p_selected.shape)  # 输出形状查看
# 获取 coor 中对应的点
coor_selected = np.array([coor[i, id_param[i], :] for i in range(coor.shape[0])])  # 形状为 (100, N, 2)

# 在第二个维度上拼接 coor 和 u, v, p
params_u = np.concatenate((coor_selected, np.expand_dims(u_selected, -1)), axis=-1)  # 形状为 (100, N, 3)
params_v = np.concatenate((coor_selected, np.expand_dims(v_selected, -1)), axis=-1)  # 形状为 (100, N, 3)
params_p = np.concatenate((coor_selected, np.expand_dims(p_selected, -1)), axis=-1)  # 形状为 (100, N, 3)

# 打印输出形状
print(params_u.shape, params_v.shape, params_p.shape)  # 输出形状查看



In [ ]:
# define dataset as torch.tensor 将数据集定义为torch.tensor
params_u = torch.tensor(params_u)    # (B, N, 3)
params_v = torch.tensor(params_v) 
params_p = torch.tensor(params_p) 
u = torch.tensor(u)    # (B, M)
v = torch.tensor(v)    # (B, N)
p = torch.tensor(p) 
flag_ID = torch.tensor(flag_ID)
coor = torch.tensor(coor)    # (M,2)
flag_BCxy = torch.tensor(flag_BCxy) 
flag_load = torch.tensor(flag_load) 
Times = torch.tensor(Times)
print(params_u.shape)
print(params_v.shape)
print(params_p.shape)

print(u.shape)
print(v.shape)
print(p.shape)
print(coor.shape)
print(Times.shape)

In [ ]:
# 生成随机索引
torch.manual_seed(42)
indices = torch.randperm(datasize)  # 随机生成 [0, datasize) 的打乱索引

# 根据随机索引重新排列数据
params_u = params_u[indices]
params_v = params_v[indices]
params_p = params_p[indices]
u = u[indices]
v = v[indices]
p = p[indices]
coor = coor[indices]
flag_BCxy = flag_BCxy[indices]
flag_load = flag_load[indices]
flag_ID = flag_ID[indices]
Times = Times[indices]
# 划分数据集
bar1 = [0, int(0.9 * datasize)]  # 前 70% 为训练集
bar2 = [int(0.9 * datasize), int(0.95 * datasize)]  # 10% 为验证集
bar3 = [int(0.95 * datasize), datasize]  # 剩余 20% 为测试集

# 添加 coor, flag_BCxy, flag_BCy, flag_load 到 train_dataset 中
train_dataset = torch.utils.data.TensorDataset(
    params_u[bar1[0]:bar1[1], :, :], 
    params_v[bar1[0]:bar1[1], :, :], 
    params_p[bar1[0]:bar1[1], :, :], 
    u[bar1[0]:bar1[1], :], 
    v[bar1[0]:bar1[1], :], 
    p[bar1[0]:bar1[1], :],
    coor[bar1[0]:bar1[1], :, :],  # 添加 coor
    flag_BCxy[bar1[0]:bar1[1], :],  # 添加 flag_BCxy
    flag_load[bar1[0]:bar1[1], :],   # 添加 flag_load
    Times[bar1[0]:bar1[1], :]
)

val_dataset = torch.utils.data.TensorDataset(
    params_u[bar2[0]:bar2[1], :, :], 
    params_v[bar2[0]:bar2[1], :, :], 
    params_p[bar2[0]:bar2[1], :, :], 
    u[bar2[0]:bar2[1], :], 
    v[bar2[0]:bar2[1], :], 
    p[bar2[0]:bar2[1], :],
    coor[bar2[0]:bar2[1], :, :],  # 添加 coor
    flag_BCxy[bar2[0]:bar2[1], :],  # 添加 flag_BCxy
    flag_load[bar2[0]:bar2[1], :],   # 添加 flag_load
    Times[bar2[0]:bar2[1], :]
)

test_dataset = torch.utils.data.TensorDataset(
    params_u[bar3[0]:bar3[1], :, :], 
    params_v[bar3[0]:bar3[1], :, :], 
    params_p[bar3[0]:bar3[1], :, :], 
    u[bar3[0]:bar3[1], :], 
    v[bar3[0]:bar3[1], :], 
    p[bar3[0]:bar3[1], :],
    coor[bar3[0]:bar3[1], :, :],  # 添加 coor
    flag_BCxy[bar3[0]:bar3[1], :],  # 添加 flag_BCxy
    flag_load[bar3[0]:bar3[1], :],  # 添加 flag_load
    Times[bar3[0]:bar3[1], :],
    flag_ID[bar3[0]:bar3[1]]        # 添加 flag_ID
)

# 创建 DataLoader
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=config['train']['batchsize'], shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=config['train']['batchsize'], shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=config['train']['batchsize'], shuffle=False)

In [ ]:
import torch
import torch.nn as nn

class fc(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.ff = nn.Linear(dim, dim)
        self.act = nn.Tanh()

    def forward(self, x):
        return self.act(self.ff(x))

In [ ]:
class DeepMONet(nn.Module):
    def __init__(self, config):
        """
        mode: 'u', 'v', 'p' 分别表示不同的模型
        """
        mode_u = 'u'  # 对应模式u
        mode_v = 'v'  # 对应模式v
        mode_p = 'p'  # 对应模式p
        super().__init__()
        self.mode_u = mode_u
        self.mode_v = mode_v
        self.mode_p = mode_p
        # branch network 1
        trunk_layers_u = [nn.Linear(3, config['model']['fc_dim']), nn.Tanh()]
        for _ in range(config['model']['N_layer'] - 1):
            trunk_layers_u.append(nn.Linear(config['model']['fc_dim'], config['model']['fc_dim']))
            trunk_layers_u.append(nn.Tanh())
        trunk_layers_u.append(nn.Linear(config['model']['fc_dim'], config['model']['fc_dim']))
        self.branch_u = nn.Sequential(*trunk_layers_u)

        trunk_layers_v = [nn.Linear(3, config['model']['fc_dim']), nn.Tanh()]
        for _ in range(config['model']['N_layer'] - 1):
            trunk_layers_v.append(nn.Linear(config['model']['fc_dim'], config['model']['fc_dim']))
            trunk_layers_v.append(nn.Tanh())
        trunk_layers_v.append(nn.Linear(config['model']['fc_dim'], config['model']['fc_dim']))
        self.branch_v = nn.Sequential(*trunk_layers_v)

        trunk_layers_p = [nn.Linear(3, config['model']['fc_dim']), nn.Tanh()]
        for _ in range(config['model']['N_layer'] - 1):
            trunk_layers_p.append(nn.Linear(config['model']['fc_dim'], config['model']['fc_dim']))
            trunk_layers_p.append(nn.Tanh())
        trunk_layers_p.append(nn.Linear(config['model']['fc_dim'], config['model']['fc_dim']))
        self.branch_p = nn.Sequential(*trunk_layers_p)

         # trunk network  
        def fc_layers(fc_dim):
            layers = [nn.Linear(fc_dim, fc_dim)]
            return nn.Sequential(*layers)
        self.FC1 = nn.Linear(3, config['model']['fc_dim'])
        self.trunk_layers = nn.ModuleList(
            [nn.ModuleList([fc_layers(config['model']['fc_dim'])
                            for _ in range(config['model'][f'N_layer_{mode}'])]) 
             for mode in ['u', 'v', 'p']]
        )

        self.act = nn.Tanh()
    
    def forward(self, x_coor, y_coor, Times, par_u, par_v, par_p):
        
        xy = torch.cat((x_coor.unsqueeze(-1), y_coor.unsqueeze(-1),Times.unsqueeze(-1)), -1)  # (B, M, 2)
        enc_u = self.branch_u(par_u)
        enc_u = torch.amax(enc_u, 1, keepdim=True)
        u = self.FC1(xy)
        u = self.act(u)
          
        enc_v = self.branch_v(par_v)
        enc_v = torch.amax(enc_v, 1, keepdim=True)
        v = self.FC1(xy)
        v = self.act(v)
          
        enc_p = self.branch_p(par_p)
        enc_p = torch.amax(enc_p, 1, keepdim=True)
        p = self.FC1(xy)
        p = self.act(p)
        if self.mode_u == 'u':  
            for i, trunk in enumerate(self.trunk_layers[0]):
                u = u * enc_u
                u = trunk(u)
                if i < len(self.trunk_layers[0]) - 1:  
                    u = self.act(u)      
            u = torch.mean(u * enc_u, -1)  # (B, M)

        if self.mode_v == 'v':
            for i, trunk in enumerate(self.trunk_layers[1]):
                v = v * enc_v
                v = trunk(v)
                if i < len(self.trunk_layers[1]) - 1:  
                    v = self.act(v)      
            v = torch.mean(v * enc_v, -1)  # (B, M)

        if self.mode_p == 'p':
            for i, trunk in enumerate(self.trunk_layers[2]):
                p = p * enc_p
                p = trunk(p)
                if i < len(self.trunk_layers[2]) - 1:  
                    p = self.act(p)      
            p = torch.mean(p * enc_p, -1)  # (B, M)
        return u, v, p
          
  

In [ ]:

def val(model, loader, device, config):
    """
    验证函数 - 支持分批处理以避免内存溢出
    config: 训练配置参数
    """
    mean_relative_L2 = 0
    mean_relative_L2_u = 0
    mean_relative_L2_v = 0
    mean_relative_L2_p = 0
    total_val_loss_u = 0
    total_val_loss_v = 0
    total_val_loss_p = 0
    num = 0
    num_u = 0
    num_v = 0
    num_p = 0
    mse = nn.MSELoss()

    model.eval()
    with torch.no_grad():
        for (par_u, par_v, par_p, u, v, p, coors,  flag_BCxy, flag_load, Times) in loader:
            
            par_u = par_u.float().to(device)
            par_v = par_v.float().to(device)
            par_p = par_p.float().to(device)
            
            for _ in range(config['train']['coor_sampling_freq']):
                # 获取满足条件的点索引（与训练函数相同的筛选条件）
                # valid_indices = np.arange(flag_BCxy.shape[0])
                valid_indices = np.where((flag_BCxy == 0) & (flag_load == 0))[1]
                # 限制采样点数以避免内存溢出（使用coor_sampling_size）
                max_points = config['train']['coor_sampling_size']
                if len(valid_indices) > max_points:
                    # 随机采样指定数量的点
                    sampled_indices = np.random.choice(valid_indices, size=max_points, replace=False)
                else:
                    sampled_indices = valid_indices

                # 准备数据（使用采样的点）
                test_coor_x = coors[:, sampled_indices, 0].float().to(device)
                test_coor_y = coors[:, sampled_indices, 1].float().to(device)
                test_Times = Times[:, sampled_indices].float().to(device)
                u_valid = u[:, sampled_indices].float().to(device)
                v_valid = v[:, sampled_indices].float().to(device)
                p_valid = p[:, sampled_indices].float().to(device)
                
                # model forward
                u_pred, v_pred, p_pred = model(test_coor_x, test_coor_y, test_Times, par_u, par_v, par_p)
                
                # 计算误差
                L2_relative = torch.sqrt(torch.sum((u_pred-u_valid)**2 + (v_pred-v_valid)**2 + (p_pred-p_valid)**2, -1)) / (torch.sqrt(torch.sum((u_valid)**2+ (v_valid)**2+ (p_valid)**2 , -1)) + 1e-8)
                L2_relative_u = torch.sqrt(torch.sum((u_pred-u_valid)**2 , -1)) / (torch.sqrt(torch.sum((u_valid)**2 , -1)))
                L2_relative_v = torch.sqrt(torch.sum((v_pred-v_valid)**2 , -1)) / (torch.sqrt(torch.sum((v_valid)**2 , -1)))
                L2_relative_p = torch.sqrt(torch.sum((p_pred-p_valid)**2 , -1)) / (torch.sqrt(torch.sum((p_valid)**2 , -1)))

                val_loss_u = mse(u_pred, u_valid)
                val_loss_v = mse(v_pred, v_valid)
                val_loss_p = mse(p_pred, p_valid)

                # 累加每个批次的损失
                total_val_loss_u += val_loss_u.detach().cpu().item()
                total_val_loss_v += val_loss_v.detach().cpu().item()
                total_val_loss_p += val_loss_p.detach().cpu().item()


                # compute relative error（转换为Python数值）
                mean_relative_L2 += torch.sum(L2_relative).detach().cpu().item()
                num += u_valid.shape[0] 
                
                mean_relative_L2_u += torch.sum(L2_relative_u).detach().cpu().item()
                num_u += u_valid.shape[0]
                mean_relative_L2_v += torch.sum(L2_relative_v).detach().cpu().item()
                num_v += u_valid.shape[0]
                mean_relative_L2_p += torch.sum(L2_relative_p).detach().cpu().item()
                num_p += u_valid.shape[0]
                
                # 清理内存
                del u_pred, v_pred, p_pred
                del u_valid, v_valid, p_valid
                del test_coor_x, test_coor_y
                del L2_relative, L2_relative_u, L2_relative_v, L2_relative_p
            
            # 定期清理GPU缓存
            torch.cuda.empty_cache()
    
    # 计算平均误差
    if num > 0:
        mean_relative_L2 = mean_relative_L2 / num
        mean_relative_L2_u = mean_relative_L2_u / num_u
        mean_relative_L2_v = mean_relative_L2_v / num_v
        mean_relative_L2_p = mean_relative_L2_p / num_p
    else:
        mean_relative_L2 = 0.0
        mean_relative_L2_u = 0.0
        mean_relative_L2_v = 0.0
        mean_relative_L2_p = 0.0

    # 计算平均损失
    if num > 0:
        val_loss_u = total_val_loss_u / num_u
        val_loss_v = total_val_loss_v / num_v
        val_loss_p = total_val_loss_p / num_p

    else:
        val_loss_u = 0.0
        val_loss_v = 0.0
        val_loss_p = 0.0


    return mean_relative_L2_u, mean_relative_L2_v, mean_relative_L2_p, mean_relative_L2, val_loss_u, val_loss_v, val_loss_p

In [ ]:
import csv

def save_relative_errors(train_loss, val_loss, train_errors,errors, epochs, filename, config, args):
    # 尝试读取现有的文件，如果文件不存在，则初始化为空
    try:
        with open(filename, 'r', newline='') as f:
            reader = csv.reader(f)
            # 跳过标题
            next(reader)
            # 获取 Epochs 列的所有值
            existing_epochs = [int(row[0]) for row in reader if row]  # 只读取非空行
    except FileNotFoundError:
        # 如果文件不存在，初始化 existing_epochs 为空列表
        existing_epochs = []

    visual_freq = config['train']['visual_freq']
    # 确保追加的 Epochs 是从已有的最大值开始
    if existing_epochs:
        last_epoch = existing_epochs[-1]  # 获取文件中最后一个 Epoch 值
        n =  config['train']['visual_freq']
    else:
        last_epoch = config['train']['visual_freq']  # 如果文件为空或不存在
        n = 0

    # 生成新的 Epochs，根据传入的 epochs 递增
    new_epochs = [last_epoch + (i + n) for i in epochs]   
    
    # 将数据追加到文件
    with open(filename, 'a', newline='') as f:
        writer = csv.writer(f)

        # 如果文件为空，写入标题行
        if not existing_epochs:
            writer.writerow(['Epoch', 'err_u', 'err_v', 'err_p', 'err','train_err_u', 'train_err_v', 'train_err_p', 'train_err'])  # 写入标题行
        
        # 在写入新数据之前，先写入一个空行
        if existing_epochs:
            writer.writerow([])  # 空行
        # 写入新的 Epochs 和相应的 Relative_Error 数据
        for epoch, (err_u, err_v, err_p, err), (mean_train_L2_u, mean_train_L2_v, mean_train_L2_p, mean_train_L2), (train_loss_u, train_loss_v, train_loss_p), (val_loss_u, val_loss_v, val_loss_p) in zip(new_epochs, errors, train_errors,train_loss,val_loss):
            writer.writerow([epoch, err_u, err_v, err_p, err,mean_train_L2_u, mean_train_L2_v, mean_train_L2_p, mean_train_L2, train_loss_u, train_loss_v, train_loss_p, val_loss_u, val_loss_v, val_loss_p])  # 写入新数据
            


In [ ]:
def plot_loss_from_file(file_path):
    # 读取文件数据，假设文件格式是: Epochs, Relative_Error
    data = np.loadtxt(file_path, delimiter=',', skiprows=1)  # 跳过第一行标题
    
    # # 提取验证集误差
    epochs = data[:, 0]
    # err_u = data[:, 1]
    # err_v = data[:, 2]
    # err_p = data[:, 3]
    # err = data[:, 4]
    
    # # 提取训练集误差
    # train_err_u = data[:, 5]
    # train_err_v = data[:, 6]
    # train_err_p = data[:, 7]
    # train_err = data[:, 8]

    train_loss_u = data[:, 9]
    train_loss_v = data[:, 10]
    train_loss_p = data[:, 11]

    val_loss_u = data[:, 12]
    val_loss_v = data[:, 13]
    val_loss_p = data[:, 14]

    
    
    # 绘制图形
    plt.figure(figsize=(12, 8))
    
    # 验证集曲线
    plt.plot(epochs, train_loss_u, marker='o', label="val_err_u", color='r', linestyle='-')
    plt.plot(epochs, train_loss_v, marker='s', label="val_err_v", color='g', linestyle='-')
    plt.plot(epochs, train_loss_p, marker='^', label="val_err_p", color='b', linestyle='-')
 
    
    # 训练集曲线（用虚线区分）
    plt.plot(epochs, val_loss_u, marker='o', label="train_err_u", color='r', linestyle='--')
    plt.plot(epochs, val_loss_v, marker='s', label="train_err_v", color='g', linestyle='--')
    plt.plot(epochs, val_loss_p, marker='^', label="train_err_p", color='b', linestyle='--')
 
    
    # 图形样式
    plt.title("Training vs Validation Loss", fontsize=14)
    plt.xlabel("Epochs", fontsize=12)
    plt.ylabel("Loss Error", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(fontsize=10, ncol=2)  # 分两列显示，防止图例过长

    # 保存并展示图片
    plt.tight_layout()
    plt.savefig('loss.png', dpi=300)
    plt.show()
    print("✅ Plot (including training & validation errors) saved as 'Loss.png'")


In [ ]:
def plot_relative_errors_from_file(file_path):
    # 读取文件数据，假设文件格式是: Epochs, Relative_Error
    data = np.loadtxt(file_path, delimiter=',', skiprows=1)  # 跳过第一行标题
    
    # 提取验证集误差
    epochs = data[:, 0]
    err_u = data[:, 1]
    err_v = data[:, 2]
    err_p = data[:, 3]
    err = data[:, 4]
    
    # 提取训练集误差
    train_err_u = data[:, 5]
    train_err_v = data[:, 6]
    train_err_p = data[:, 7]
    train_err = data[:, 8]

    train_loss_u = data[:, 9]
    train_loss_v = data[:, 10]
    train_loss_p = data[:, 11]

    val_loss_u = data[:, 12]
    val_loss_v = data[:, 13]
    val_loss_p = data[:, 14]

    
    
    # 绘制图形
    plt.figure(figsize=(12, 8))
    
    # 验证集曲线
    plt.plot(epochs, err_u, marker='o', label="val_err_u", color='r', linestyle='-')
    plt.plot(epochs, err_v, marker='s', label="val_err_v", color='g', linestyle='-')
    plt.plot(epochs, err_p, marker='^', label="val_err_p", color='b', linestyle='-')
    plt.plot(epochs, err, marker='x', label="val_err_total", color='m', linestyle='-')
    
    # 训练集曲线（用虚线区分）
    plt.plot(epochs, train_err_u, marker='o', label="train_err_u", color='r', linestyle='--')
    plt.plot(epochs, train_err_v, marker='s', label="train_err_v", color='g', linestyle='--')
    plt.plot(epochs, train_err_p, marker='^', label="train_err_p", color='b', linestyle='--')
    plt.plot(epochs, train_err, marker='x', label="train_err_total", color='m', linestyle='--')
    
    # 图形样式
    plt.title("Training vs Validation L2 Relative Errors", fontsize=14)
    plt.xlabel("Epochs", fontsize=12)
    plt.ylabel("L2 Relative Error", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(fontsize=10, ncol=2)  # 分两列显示，防止图例过长

    # 保存并展示图片
    plt.tight_layout()
    plt.savefig('L2_relative_error.png', dpi=300)
    plt.show()
    print("✅ Plot (including training & validation errors) saved as 'relative_error.png'")


In [ ]:
# function for testing
def test(model, loader, device, args):
 
    mean_relative_L2 = 0
    num = 0
    max_relative_err = -1
    min_relative_err = np.inf

    mean_relative_L2_u = 0
    num_u = 0
    max_relative_err_u = -1
    min_relative_err_u = np.inf

    mean_relative_L2_v = 0
    num_v = 0
    max_relative_err_v = -1
    min_relative_err_v = np.inf

    mean_relative_L2_p = 0
    num_p = 0
    max_relative_err_p = -1
    min_relative_err_p = np.inf
    
    target_id = args.target_id  # 你想查找的 flag_ID
    found_ID = None
    found_u, found_v, found_p = None, None, None  

    for (par_u, par_v, par_p, u, v, p, coors, flag_BCxy, flag_load,  Times, flag_ID) in loader:
    
        # split the coordinates
        test_coor_x = coors[:,  :, 0].unsqueeze(0).float().to(device).squeeze(0)
        test_coor_y = coors[:,  :, 1].unsqueeze(0).float().to(device).squeeze(0)
        coors = coors[:,  :, :].float().to(device)
        u = u[:, :].float().to(device)
        v = v[:, :].float().to(device)
        p = p[:, :].float().to(device)
        Times = Times[:, :].float().to(device)
        # move the data to device
        batch = par_u.shape[0]
        par_u = par_u.float().to(device)
        flag_ID = flag_ID.float().to(device)
        par_v = par_v.float().to(device)
        par_p = par_p.float().to(device)


        # model forward
        u_pred, v_pred, p_pred = model(test_coor_x, test_coor_y,Times, par_u, par_v, par_p)
        
        
        L2_relative = torch.sqrt(torch.sum((u_pred-u)**2 + (v_pred-v)**2 + (p_pred-p)**2, -1)) / torch.sqrt(torch.sum((u)**2 + (v)**2 + (p)**2, -1))
        L2_relative_u = torch.sqrt(torch.sum((u_pred-u)**2 , -1)) / torch.sqrt(torch.sum((u)**2 , -1))
        L2_relative_v = torch.sqrt(torch.sum((v_pred-v)**2 , -1)) / torch.sqrt(torch.sum((v)**2 , -1))
        L2_relative_p = torch.sqrt(torch.sum((p_pred-p)**2, -1)) / torch.sqrt(torch.sum((p)**2, -1))
        
        max_err, max_err_idx = torch.topk(L2_relative, 1)
        max_err_u, max_err_idx_u = torch.topk(L2_relative_u, 1)
        max_err_v, max_err_idx_v = torch.topk(L2_relative_v, 1)
        max_err_p, max_err_idx_p = torch.topk(L2_relative_p, 1)

        if max_err > max_relative_err:
            max_relative_err = max_err
        min_err, min_err_idx = torch.topk(-L2_relative, 1)
        min_err = -min_err
        if min_err < min_relative_err:
            min_relative_err = min_err
        mean_relative_L2 += torch.sum(L2_relative).detach().cpu().item()
        num += u.shape[0]

        print(flag_ID)
        #查找想到的ID
        if int(flag_ID[max_err_idx_u].detach().cpu().numpy().item()) == int(target_id):
            found_ID = int(flag_ID[max_err_idx_u].detach().cpu().numpy().item())
            found_ID_xcoor = coors[max_err_idx_u,:,0].squeeze(0).squeeze(-1).detach().cpu().numpy()
            found_ID_ycoor = coors[max_err_idx_u,:,1].squeeze(0).squeeze(-1).detach().cpu().numpy()
            found_u = u[max_err_idx_u].detach().cpu().numpy()  # 转换为 NumPy
            found_v = v[max_err_idx_u].detach().cpu().numpy()
            found_p = p[max_err_idx_u].detach().cpu().numpy()
            found_u_pred = u_pred[max_err_idx_u].detach().cpu().numpy()  # 转换为 NumPy
            found_v_pred = v_pred[max_err_idx_u].detach().cpu().numpy()
            found_p_pred = p_pred[max_err_idx_u].detach().cpu().numpy()
            print(f"找到样本 ID: {target_id}")
            
                
        
        if max_err_u > max_relative_err_u:
             max_relative_err_u = max_err_u
             worst_xcoor_u = coors[max_err_idx,:,0].squeeze(0).squeeze(-1).detach().cpu().numpy()
             worst_ycoor_u = coors[max_err_idx,:,1].squeeze(0).squeeze(-1).detach().cpu().numpy()
             worst_flag_ID_u = flag_ID[max_err_idx_u].detach().cpu().numpy()
             worst_f_u = u_pred[max_err_idx_u, :].detach().cpu().numpy()
             worst_gt_u = u[max_err_idx_u, :].detach().cpu().numpy()
             # 反归一化
             worst_f_u = worst_f_u * (max_u - min_u) + min_u
             worst_gt_u = worst_gt_u * (max_u - min_u) + min_u
        min_err_u, min_err_idx_u = torch.topk(-L2_relative_u, 1)
        min_err_u = -min_err_u
        if min_err_u < min_relative_err_u:
             min_relative_err_u = min_err_u
             best_xcoor_u = coors[min_err_idx,:,0].squeeze(0).squeeze(-1).detach().cpu().numpy()
             best_ycoor_u = coors[min_err_idx,:,1].squeeze(0).squeeze(-1).detach().cpu().numpy()
             best_flag_ID_u = flag_ID[min_err_idx_u].detach().cpu().numpy()
             best_f_u = u_pred[min_err_idx_u,:].detach().cpu().numpy()
             best_gt_u = u[min_err_idx_u,:].detach().cpu().numpy()
             # 反归一化
             best_f_u = best_f_u * (max_u - min_u) + min_u
             best_gt_u = best_gt_u * (max_u - min_u) + min_u
        mean_relative_L2_u += torch.sum(L2_relative_u).detach().cpu().item()
        num_u += u.shape[0]

        if max_err_v > max_relative_err_v:
            max_relative_err_v = max_err_v
            worst_xcoor_v = coors[max_err_idx_v,:,0].squeeze(0).squeeze(-1).detach().cpu().numpy()
            worst_ycoor_v = coors[max_err_idx_v,:,1].squeeze(0).squeeze(-1).detach().cpu().numpy()
            worst_flag_ID_v = flag_ID[max_err_idx_v].detach().cpu().numpy()
            worst_f_v = v_pred[max_err_idx_v,:].detach().cpu().numpy()
            worst_gt_v = v[max_err_idx_v,:].detach().cpu().numpy()
            worst_f_v = worst_f_v * (max_v - min_v) + min_v
            worst_gt_v = worst_gt_v * (max_v - min_v) + min_v
        min_err_v, min_err_idx_v = torch.topk(-L2_relative_v, 1)
        min_err_v = -min_err_v
        if min_err_v < min_relative_err_v:
            min_relative_err_v = min_err_v
            best_xcoor_v = coors[min_err_idx_v,:,0].squeeze(0).squeeze(-1).detach().cpu().numpy()
            best_ycoor_v = coors[min_err_idx_v,:,1].squeeze(0).squeeze(-1).detach().cpu().numpy()
            best_flag_ID_v = flag_ID[min_err_idx_v].detach().cpu().numpy()
            best_f_v = v_pred[min_err_idx_v,:].detach().cpu().numpy()
            best_gt_v = v[min_err_idx_v,:].detach().cpu().numpy()
            # 反归一化
            best_f_v = best_f_v * (max_v - min_v) + min_v
            best_gt_v = best_gt_v * (max_v - min_v) + min_v
        mean_relative_L2_v += torch.sum(L2_relative_v).detach().cpu().item()
        num_v += v.shape[0]
            
        if max_err_p > max_relative_err_p:
            max_relative_err_p = max_err_p
            worst_xcoor_p = coors[max_err_idx_p,:,0].squeeze(0).squeeze(-1).detach().cpu().numpy()
            worst_ycoor_p = coors[max_err_idx_p,:,1].squeeze(0).squeeze(-1).detach().cpu().numpy()
            worst_flag_ID_p = flag_ID[max_err_idx_p].detach().cpu().numpy()
            worst_f_p = p_pred[max_err_idx_p,:].detach().cpu().numpy()
            worst_gt_p = p[max_err_idx_p,:].detach().cpu().numpy()
            # 反归一化
            worst_f_p = worst_f_p * (max_p - min_p) + min_p
            worst_gt_p = worst_gt_p * (max_p - min_p) + min_p
        min_err_p, min_err_idx_p = torch.topk(-L2_relative_p, 1)
        min_err_p = -min_err_p
        if min_err_p < min_relative_err_p:
            min_relative_err_p = min_err_p
            best_xcoor_p = coors[min_err_idx_p,:,0].squeeze(0).squeeze(-1).detach().cpu().numpy()
            best_ycoor_p = coors[min_err_idx_p,:,1].squeeze(0).squeeze(-1).detach().cpu().numpy()
            best_flag_ID_p = flag_ID[min_err_idx_p].detach().cpu().numpy()
            best_f_p = p_pred[min_err_idx_p,:].detach().cpu().numpy()
            best_gt_p = p[min_err_idx_p,:].detach().cpu().numpy()
            # 反归一化
            best_f_p = best_f_p * (max_p - min_p) + min_p
            best_gt_p = best_gt_p * (max_p - min_p) + min_p
        mean_relative_L2_p += torch.sum(L2_relative_p).detach().cpu().item()
        num_p += p.shape[0]

        one_flag_ID = int(flag_ID[max_err_idx_u].detach().cpu().numpy().item())
        one_relative_L2_u = torch.sum(L2_relative_u).detach().cpu().item()
        one_num_u = u.shape[0]
        one_relative_L2_v = torch.sum(L2_relative_v).detach().cpu().item()
        one_num_v = v.shape[0]
        one_relative_L2_p = torch.sum(L2_relative_p).detach().cpu().item()
        one_num_p = p.shape[0]
        one_relative_L2_u /= one_num_u
        one_relative_L2_v /= one_num_v
        one_relative_L2_p /= one_num_p
        
        file_path = "relative_L2.txt"
        # 计算 batch 级别的均值
        one_relative_L2_u = torch.sum(L2_relative_u).detach().cpu().item()
        one_num_u = u.shape[0]
        one_relative_L2_v = torch.sum(L2_relative_v).detach().cpu().item()
        one_num_v = v.shape[0]
        one_relative_L2_p = torch.sum(L2_relative_p).detach().cpu().item()
        one_num_p = p.shape[0]
        
        one_relative_L2_u /= one_num_u
        one_relative_L2_v /= one_num_v
        one_relative_L2_p /= one_num_p
        
        # 遍历整个 batch（每个样本都写入文件）
        for idx in range(flag_ID.shape[0]):  
            one_flag_ID = int(flag_ID[idx].detach().cpu().numpy().item())  # 取当前样本的 ID
            one_sample_L2_u = L2_relative_u[idx].detach().cpu().item()  # 取当前样本的 U 误差
            one_sample_L2_v = L2_relative_v[idx].detach().cpu().item()  # 取当前样本的 V 误差
            one_sample_L2_p = L2_relative_p[idx].detach().cpu().item()  # 取当前样本的 P 误差
        
            with open(file_path, "a") as f:
                f.write(f"{one_flag_ID} {one_relative_L2_u} {one_relative_L2_v} {one_relative_L2_p}\n")


    mean_relative_L2 /= num
    mean_relative_L2_u /= num_u
    mean_relative_L2_v /= num_v
    mean_relative_L2_p /= num_p
    
    # make the coordinates to numpy
    # coor_x = test_coor_x[0].detach().cpu().numpy()
    # coor_y = test_coor_y[0].detach().cpu().numpy()

    # color bar range
    max_color_u = np.amax(best_gt_u)
    min_color_u = np.amin(best_gt_u)
    max_color_worst_u = np.amax(worst_gt_u)
    min_color_worst_u = np.amin(worst_gt_u)
    # max_color_r_u = np.amax([np.amax(worst_gt_u-worst_f_u), np.amax(best_gt_u-best_f_u)])
    # min_color_r_u = np.amin([np.amin(worst_gt_u-worst_f_u), np.amin(best_gt_u-best_f_u)])
    max_color_r_u = np.amax([np.amax(np.abs(worst_gt_u - worst_f_u)), np.amax(np.abs(best_gt_u - best_f_u))])
    min_color_r_u = np.amin([np.amin(np.abs(worst_gt_u - worst_f_u)), np.amin(np.abs(best_gt_u - best_f_u))])

    max_color_v = np.amax(best_gt_v)
    min_color_v = np.amin(best_gt_v)
    max_color_worst_v = np.amax(worst_gt_v)
    min_color_worst_v = np.amin(worst_gt_v)
    max_color_r_v = np.amax([np.amax(np.abs(worst_gt_v - worst_f_v)), np.amax(np.abs(best_gt_v - best_f_v))])
    min_color_r_v = np.amin([np.amin(np.abs(worst_gt_v - worst_f_v)), np.amin(np.abs(best_gt_v - best_f_v))])

    max_color_p = np.amax(best_gt_p)
    min_color_p = np.amin(best_gt_p)
    max_color_worst_p = np.amax(worst_gt_p)
    min_color_worst_p = np.amin(worst_gt_p)
    max_color_r_p = np.amax([np.amax(np.abs(worst_gt_p - worst_f_p)), np.amax(np.abs(best_gt_p - best_f_p))])
    min_color_r_p = np.amin([np.amin(np.abs(worst_gt_p - worst_f_p)), np.amin(np.abs(best_gt_p - best_f_p))])


    if found_ID == int(target_id):
        max_found_u = np.amax(found_u)
        min_found_u = np.amin(found_u)
        max_found_r_u = np.amax(np.abs(found_u - found_u_pred))
        max_found_v = np.amax(found_v)
        min_found_v = np.amin(found_v)
        max_found_r_v = np.amax(np.abs(found_v - found_v_pred))
        max_found_p = np.amax(found_p)
        min_found_p = np.amin(found_p)
        max_found_r_p = np.amax(np.abs(found_p - found_p_pred))
    # make a plot - 提升视觉质量
    SS = 8  # 增大点的大小以提升视觉效果
    # 使用更现代的配色方案：对于预测和真值使用viridis，对于误差使用plasma
    cm_pred = plt.cm.get_cmap('viridis')
    cm_error = plt.cm.get_cmap('plasma')
    
    from mpl_toolkits.axes_grid1 import make_axes_locatable
    
    # u #################################################################################################################
    fig, axes = plt.subplots(2, 3, figsize=(18, 10), dpi=300, facecolor='white')  # 创建 2x3 的子图网格
    fig.suptitle('Velocity Component U - Comparison Analysis', fontsize=18, fontweight='bold', y=0.98)
    
    titles = [
        f'Prediction (Worst Case, ID: {worst_flag_ID_u})',
        f'Ground Truth (Worst Case, ID: {worst_flag_ID_u})',
        f'Absolute Error (Worst Case, ID: {worst_flag_ID_u})',
        f'Prediction (Best Case, ID: {best_flag_ID_u})',
        f'Ground Truth (Best Case, ID: {best_flag_ID_u})',
        f'Absolute Error (Best Case, ID: {best_flag_ID_u})'
    ]

    datasets = [
        worst_f_u, worst_gt_u, np.abs(worst_f_u - worst_gt_u),
        best_f_u, best_gt_u, np.abs(best_f_u - best_gt_u)
    ]
    color_ranges = [
        (min_color_worst_u, max_color_worst_u),
        (min_color_worst_u, max_color_worst_u),
        (0, max_color_r_u),
        (min_color_u, max_color_u),
        (min_color_u, max_color_u),
        (0, max_color_r_u)
    ]
    
    for i, ax in enumerate(axes.flatten()):
        data = datasets[i]
        vmin, vmax = color_ranges[i]
        # 误差图使用不同的colormap
        cmap = cm_error if i in [2, 5] else cm_pred
        
        if i < 3:
            scatter = ax.scatter(worst_xcoor_u, worst_ycoor_u, c=data, cmap=cmap, vmin=vmin, vmax=vmax, 
                               marker='o', s=SS, edgecolors='none', alpha=0.85, linewidths=0)
        else:
            scatter = ax.scatter(best_xcoor_u, best_ycoor_u, c=data, cmap=cmap, vmin=vmin, vmax=vmax, 
                               marker='o', s=SS, edgecolors='none', alpha=0.85, linewidths=0)
    
        # 添加自定义的颜色条，改进样式
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="6%", pad=0.08)
        cbar = fig.colorbar(scatter, cax=cax)
        cbar.ax.tick_params(labelsize=9, width=1, length=4)
        cbar.outline.set_linewidth(1)
        cbar.ax.set_ylabel('Value', rotation=270, labelpad=15, fontsize=10, fontweight='bold')
    
        ax.set_title(titles[i], fontsize=13, fontweight='bold', pad=10)
        ax.set_aspect('equal')
        # 添加浅色背景，关闭坐标轴
        ax.set_facecolor('#f8f9fa')
        ax.axis('off')
    
    fig.subplots_adjust(hspace=0.15, wspace=0.25, top=0.92, bottom=0.08, left=0.05, right=0.95)
    plt.savefig('u_plots.png', dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.show()
    
    # v #################################################################################################################
    fig, axes = plt.subplots(2, 3, figsize=(18, 10), dpi=300, facecolor='white')  # 创建 2x3 的子图网格
    fig.suptitle('Velocity Component V - Comparison Analysis', fontsize=18, fontweight='bold', y=0.98)
    
    titles = [
        f'Prediction (Worst Case, ID: {worst_flag_ID_v})',
        f'Ground Truth (Worst Case, ID: {worst_flag_ID_v})',
        f'Absolute Error (Worst Case, ID: {worst_flag_ID_v})',
        f'Prediction (Best Case, ID: {best_flag_ID_v})',
        f'Ground Truth (Best Case, ID: {best_flag_ID_v})',
        f'Absolute Error (Best Case, ID: {best_flag_ID_v})'
    ]
    datasets = [
        worst_f_v, worst_gt_v, np.abs(worst_f_v - worst_gt_v),
        best_f_v, best_gt_v, np.abs(best_f_v - best_gt_v)
    ]
    color_ranges = [
        (min_color_worst_v, max_color_worst_v),
        (min_color_worst_v, max_color_worst_v),
        (0, max_color_r_v),
        (min_color_v, max_color_v),
        (min_color_v, max_color_v),
        (0, max_color_r_v)
    ]
    
    for i, ax in enumerate(axes.flatten()):
        data = datasets[i]
        vmin, vmax = color_ranges[i]
        # 误差图使用不同的colormap
        cmap = cm_error if i in [2, 5] else cm_pred
        
        if i < 3:
            scatter = ax.scatter(worst_xcoor_v, worst_ycoor_v, c=data, cmap=cmap, vmin=vmin, vmax=vmax, 
                               marker='o', s=SS, edgecolors='none', alpha=0.85, linewidths=0)
        else:
            scatter = ax.scatter(best_xcoor_v, best_ycoor_v, c=data, cmap=cmap, vmin=vmin, vmax=vmax, 
                               marker='o', s=SS, edgecolors='none', alpha=0.85, linewidths=0)
        # 添加自定义的颜色条，改进样式
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="6%", pad=0.08)
        cbar = fig.colorbar(scatter, cax=cax)
        cbar.ax.tick_params(labelsize=9, width=1, length=4)
        cbar.outline.set_linewidth(1)
        cbar.ax.set_ylabel('Value', rotation=270, labelpad=15, fontsize=10, fontweight='bold')
    
        ax.set_title(titles[i], fontsize=13, fontweight='bold', pad=10)
        ax.set_aspect('equal')
        # 添加浅色背景，关闭坐标轴
        ax.set_facecolor('#f8f9fa')
        ax.axis('off')
    
    fig.subplots_adjust(hspace=0.15, wspace=0.25, top=0.92, bottom=0.08, left=0.05, right=0.95)
    plt.savefig('v_plots.png', dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.show()
    
    # p #################################################################################################################
    fig, axes = plt.subplots(2, 3, figsize=(18, 10), dpi=300, facecolor='white')  # 创建 2x3 的子图网格
    fig.suptitle('Pressure Component P - Comparison Analysis', fontsize=18, fontweight='bold', y=0.98)
    
    titles = [
        f'Prediction (Worst Case, ID: {worst_flag_ID_p})',
        f'Ground Truth (Worst Case, ID: {worst_flag_ID_p})',
        f'Absolute Error (Worst Case, ID: {worst_flag_ID_p})',
        f'Prediction (Best Case, ID: {best_flag_ID_p})',
        f'Ground Truth (Best Case, ID: {best_flag_ID_p})',
        f'Absolute Error (Best Case, ID: {best_flag_ID_p})'
    ]
    datasets = [
        worst_f_p, worst_gt_p, np.abs(worst_f_p - worst_gt_p),
        best_f_p, best_gt_p, np.abs(best_f_p - best_gt_p)
    ]
    color_ranges = [
        (min_color_worst_p, max_color_worst_p),
        (min_color_worst_p, max_color_worst_p),
        (0, max_color_r_p),
        (min_color_p, max_color_p),
        (min_color_p, max_color_p),
        (0, max_color_r_p)
    ]
    
    for i, ax in enumerate(axes.flatten()):
        data = datasets[i]
        vmin, vmax = color_ranges[i]
        # 误差图使用不同的colormap
        cmap = cm_error if i in [2, 5] else cm_pred
        
        if i < 3:
            scatter = ax.scatter(worst_xcoor_p, worst_ycoor_p, c=data, cmap=cmap, vmin=vmin, vmax=vmax, 
                                marker='o', s=SS, edgecolors='none', alpha=0.85, linewidths=0)
        else:
            scatter = ax.scatter(best_xcoor_p, best_ycoor_p, c=data, cmap=cmap, vmin=vmin, vmax=vmax, 
                                marker='o', s=SS, edgecolors='none', alpha=0.85, linewidths=0)
        # 添加自定义的颜色条，改进样式
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="6%", pad=0.08)
        cbar = fig.colorbar(scatter, cax=cax)
        cbar.ax.tick_params(labelsize=9, width=1, length=4)
        cbar.outline.set_linewidth(1)
        cbar.ax.set_ylabel('Value', rotation=270, labelpad=15, fontsize=10, fontweight='bold')
    
        ax.set_title(titles[i], fontsize=13, fontweight='bold', pad=10)
        ax.set_aspect('equal')
        # 添加浅色背景，关闭坐标轴
        ax.set_facecolor('#f8f9fa')
        ax.axis('off')
    
    fig.subplots_adjust(hspace=0.15, wspace=0.25, top=0.92, bottom=0.08, left=0.05, right=0.95)
    plt.savefig('p_plots.png', dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.show()

    # 目标ID #################################################################################################################

    if found_ID == int(target_id):
        ID_xcoor = found_ID_xcoor
        ID_ycoor = found_ID_ycoor
        fig, axes = plt.subplots(3, 3, figsize=(18, 16), dpi=300, facecolor='white')  # 创建 3x3 的子图网格
        fig.suptitle(f'Target ID {target_id} - Complete Analysis', fontsize=20, fontweight='bold', y=0.98)
        
        titles = [
            f'U Prediction (ID: {target_id})',
            f'U Ground Truth (ID: {target_id})',
            f'U Absolute Error (ID: {target_id})',
            f'V Prediction (ID: {target_id})',
            f'V Ground Truth (ID: {target_id})',
            f'V Absolute Error (ID: {target_id})',
            f'P Prediction (ID: {target_id})',
            f'P Ground Truth (ID: {target_id})',
            f'P Absolute Error (ID: {target_id})',
        ]
        datasets = [
            found_u_pred, found_u, np.abs(found_u - found_u_pred),
            found_v_pred, found_v, np.abs(found_v - found_v_pred),
            found_p_pred, found_p, np.abs(found_p - found_p_pred)
        ]
        
        color_ranges = [
            (min_found_u, max_found_u),
            (min_found_u, max_found_u),
            (0, max_found_r_u),
            (min_found_v, max_found_v),
            (min_found_v, max_found_v),
            (0, max_found_r_v),
            (min_found_p, max_found_p),
            (min_found_p, max_found_p),
            (0, max_found_r_p)
        ]
        
        error_indices = [2, 5, 8]  # 误差图的索引
        
        for i, ax in enumerate(axes.flatten()):
            data = datasets[i]
            vmin, vmax = color_ranges[i]
            # 误差图使用不同的colormap
            cmap = cm_error if i in error_indices else cm_pred
            
            scatter = ax.scatter(ID_xcoor, ID_ycoor, c=data, cmap=cmap, vmin=vmin, vmax=vmax, 
                               marker='o', s=SS, edgecolors='none', alpha=0.85, linewidths=0)
        
            # 添加自定义的颜色条，改进样式
            divider = make_axes_locatable(ax)
            cax = divider.append_axes("right", size="5%", pad=0.08)
            cbar = fig.colorbar(scatter, cax=cax)
            cbar.ax.tick_params(labelsize=8, width=1, length=3)
            cbar.outline.set_linewidth(1)
            cbar.ax.set_ylabel('Value', rotation=270, labelpad=12, fontsize=9, fontweight='bold')
        
            ax.set_title(titles[i], fontsize=12, fontweight='bold', pad=8)
            ax.set_aspect('equal')
            # 添加浅色背景，关闭坐标轴
            ax.set_facecolor('#f8f9fa')
            ax.axis('off')
        
        fig.subplots_adjust(hspace=0.25, wspace=0.2, top=0.95, bottom=0.05, left=0.05, right=0.95)
        plt.savefig('id_plots.png', dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
        plt.show()

    #plt.savefig(r'sample_{}_{}.png'.format(args.model_u, args.data))

    #plt.savefig(r'sample_{}_{}.png'.format(args.model_u, args.data))
    return mean_relative_L2,mean_relative_L2_u,mean_relative_L2_v,mean_relative_L2_p

In [ ]:

# define the training function
def train(args, config, model, device, loaders):

    # print training configuration
    print('training configuration')
    print('batchsize:', config['train']['batchsize'])
    print('coordinate sampling frequency:', config['train']['coor_sampling_freq'])
    print('learning rate:', config['train']['base_lr'])
    print('BC weight', config['train']['bc_weight'])
    print('PDE weight', config['train']['pde_weight'])

    # get train and test loader
    train_loader, val_loader, test_loader = loaders

    # define model training configuration 定义模型训练配置
    pbar = range(config['train']['epochs'])
    pbar = tqdm(pbar, dynamic_ncols=True, smoothing=0.1)

    # define optimizer and loss
    mse = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=config['train']['base_lr'])

    #用于评估和存储更新后的模型参数的可视频率
    vf = config['train']['visual_freq']
    
    # 使用 DataParallel 来启用多 GPU 训练
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)
    # initialize recored loss values
    avg_loss1 = np.inf
    avg_loss2 = np.inf
    avg_loss21 = np.inf
    avg_loss3 = np.inf
    avg_loss4 = np.inf
    avg_loss5 = np.inf
    avg_loss6 = np.inf
    avg_loss7 = np.inf
    avg_loss8 = np.inf
    avg_loss9 = np.inf
    
    
    # 动态权重调整参数
    error_history = []       # 存储历史误差
    adaptive_threshold = True  # 是否使用自适应阈值
    
    # 全局最差误差跟踪
    global_worst_error = 0.0
    global_worst_coords = None
    global_worst_sample_id = None

    try:
        model.load_state_dict(torch.load(f'best_model_{args.data}.pth', map_location=device))  
        
        print(" Models loaded successfully.")
    except FileNotFoundError:
        print("No pre-trained model found.")
    except Exception as e:
        print(f"Error while loading models: {e}")

    # Initialize relative error record
    val_relative_errors = []
    train_relative_errors = []
    train_loss = []
    val_loss = []
    actual_epochs = []
    # start the training
    if args.phase == 'train':
        min_val_err = np.inf
        min_val_err_u = np.inf
        min_val_err_v = np.inf
        min_val_err_p = np.inf
        
        for e in pbar:
          
            # show the performance improvement
            if e % vf == 0:
                model.eval()
                mean_train_L2 = 0
                mean_train_L2_u = 0
                mean_train_L2_v = 0
                mean_train_L2_p = 0
                mean_train_loss_u = 0
                mean_train_loss_v = 0
                mean_train_loss_p = 0          
                num = 0
                num_u = 0
                num_v = 0
                num_p = 0
                batch_count = 0
                err_u, err_v, err_p, err,  val_loss_u, val_loss_v, val_loss_p = val(model, val_loader, device, config)
                
                # relative_errors.append(err)  # 记录相对误差
                val_relative_errors.append((err_u, err_v, err_p, err))  # 添加一个元组.
                val_loss.append((val_loss_u, val_loss_v, val_loss_p))
                actual_epochs.append(e)
                
                # 更新误差历史
                error_history.append(err)
                if err_u < min_val_err_u:
                    torch.save(model.state_dict(), f'best_model_{args.data}.pth')  
                    min_val_err_u = err_u

                # update recored loss values
                avg_loss1 = 0
                avg_loss2 = 0
                avg_loss3 = 0
                avg_loss4 = 0
                avg_loss5 = 0
                avg_loss6 = 0
                avg_loss7 = 0
                avg_loss8 = 0
                avg_loss9 = 0

            # train one epoch
            model.train()
            
            for (par_u, par_v, par_p, u, v, p, coors,  flag_BCxy,  flag_load, Times) in train_loader:

                for _ in range(config['train']['coor_sampling_freq']):

                     # 获取满足条件的列索引
                    valid_indices = np.where((flag_BCxy == 0) &  (flag_load == 0))[1]
                    
                    # 限制采样点数以避免内存溢出（使用coor_sampling_size）
                    max_points = config['train']['coor_sampling_size']
                    if len(valid_indices) > max_points:
                        # 随机采样指定数量的点
                        sampled_indices = np.random.choice(valid_indices, size=max_points, replace=False)
                    else:
                        sampled_indices = valid_indices
                    
                    # prepare the data (只使用采样的点)
                    xy_BC_coors_r = coors[:,np.where(flag_BCxy==1)[1],:].float().to(device)
                    xy_BC_Times_r = Times[:,np.where(flag_BCxy==1)[1]].float().to(device)


                    load_coors_r = coors[:,np.where(flag_load==1)[1],:].float().to(device)
                    load_Times_r = Times[:,np.where(flag_load==1)[1]].float().to(device)

                    load_BC_coors_r = coors[:,sampled_indices,:].float().to(device)
                    load_Times_coors_r = Times[:,sampled_indices].float().to(device)

                    par_u = par_u.float().to(device)
                    par_v = par_v.float().to(device)
                    par_p = par_p.float().to(device)

                    # PDE #########################################################################
                    u_pred, v_pred, p_pred = model(load_BC_coors_r[:,:,0], load_BC_coors_r[:,:,1], load_Times_coors_r,par_u, par_v, par_p)
                    u_gt = u[:, sampled_indices].float().to(device)
                    v_gt = v[:, sampled_indices].float().to(device)
                    p_gt = p[:, sampled_indices].float().to(device)

                    # 边界#########################################################################
                    # 获取边界条件的索引
                    bc_indices = np.where(flag_BCxy==1)[1]
                    u_BC, v_BC, p_BC = model(xy_BC_coors_r[:,:,0], xy_BC_coors_r[:,:,1], xy_BC_Times_r,par_u, par_v, par_p)
                    u_gt_BC = u[:, bc_indices].float().to(device)
                    v_gt_BC = v[:, bc_indices].float().to(device)
                    p_gt_BC = p[:, bc_indices].float().to(device)


                    # 出口#########################################################################
                    load_indices = np.where(flag_load==1)[1]
                    u_load, v_load, p_load = model(load_coors_r[:,:,0], load_coors_r[:,:,1], load_Times_r, par_u, par_v, par_p)
                    u_gt_load = u[:, load_indices].float().to(device)
                    v_gt_load = v[:, load_indices].float().to(device)
                    p_gt_load = p[:, load_indices].float().to(device)

                    # compute the loss
                    bc_loss_u1 = mse(u_pred, u_gt)
                    bc_loss_u2 = mse(u_BC, u_gt_BC)
                    bc_loss_u4 = mse(u_load, u_gt_load)
                    total_loss_u = bc_loss_u1*20 + bc_loss_u2*1 + bc_loss_u4*10

                    bc_loss_v1 = mse(v_pred, v_gt)
                    bc_loss_v2 = mse(v_BC, v_gt_BC)
                    bc_loss_v4 = mse(v_load, v_gt_load)
                    total_loss_v = bc_loss_v1*20 + bc_loss_v2*1 + bc_loss_v4*10

                    # compute the loss
                    bc_loss_p1 = mse(p_pred, p_gt)
                    bc_loss_p2 = mse(p_BC, p_gt_BC)
                    bc_loss_p4 = mse(p_load, p_gt_load)
                    total_loss_p = bc_loss_p1*20 + bc_loss_p2*1 + + bc_loss_p4*10

                    total_loss= total_loss_u + total_loss_v + total_loss_p
                    # store the loss
                    avg_loss1 += bc_loss_u1.detach().cpu().item()
                    avg_loss2 += bc_loss_u2.detach().cpu().item()

                    
                    avg_loss4 += bc_loss_v1.detach().cpu().item()
                    avg_loss5 += bc_loss_v2.detach().cpu().item()

                    
                    avg_loss7 += bc_loss_p1.detach().cpu().item()

                    
                    # update parameter
                    optimizer.zero_grad()
                    total_loss.backward()
                    optimizer.step()

                    # 计算误差
                    L2_train = torch.sqrt(torch.sum((u_pred-u_gt)**2 + (v_pred-v_gt)**2 + (p_pred-p_gt)**2, -1)) / (torch.sqrt(torch.sum((u_gt)**2+ (v_gt)**2+ (p_gt)**2 , -1)) + 1e-8)
                    L2_train_u = torch.sqrt(torch.sum((u_pred-u_gt)**2 , -1)) / (torch.sqrt(torch.sum((u_gt)**2 , -1)))
                    L2_train_v = torch.sqrt(torch.sum((v_pred-v_gt)**2 , -1)) / (torch.sqrt(torch.sum((v_gt)**2 , -1)))
                    L2_train_p = torch.sqrt(torch.sum((p_pred-p_gt)**2 , -1)) / (torch.sqrt(torch.sum((p_gt)**2 , -1)))

                    # compute relative error（转换为Python数值）
                    mean_train_L2 += torch.sum(L2_train).detach().cpu().item()
                    num += u_gt.shape[0] 
                    
                    mean_train_L2_u += torch.sum(L2_train_u).detach().cpu().item()
                    num_u += u_gt.shape[0]
                    mean_train_L2_v += torch.sum(L2_train_v).detach().cpu().item()
                    num_v += u_gt.shape[0]
                    mean_train_L2_p += torch.sum(L2_train_p).detach().cpu().item()
                    num_p += u_gt.shape[0]

                    mean_train_loss_u += torch.sum(bc_loss_u1).detach().cpu().item()
                    mean_train_loss_v += torch.sum(bc_loss_v1).detach().cpu().item()
                    mean_train_loss_p += torch.sum(bc_loss_p1).detach().cpu().item()

                
                # 定期清理GPU缓存
                torch.cuda.empty_cache()
        
            # 计算平均误差
            if num > 0:
                mean_train_L2 = mean_train_L2 / num
                mean_train_L2_u = mean_train_L2_u / num_u
                mean_train_L2_v = mean_train_L2_v / num_v
                mean_train_L2_p = mean_train_L2_p / num_p
            else:
                mean_train_L2 = 0.0
                mean_train_L2_u = 0.0
                mean_train_L2_v = 0.0
                mean_train_L2_p = 0.0

                        # 计算平均误差
            if num > 0:
                mean_train_loss_u = mean_train_loss_u / num_u
                mean_train_loss_v = mean_train_loss_v / num_v
                mean_train_loss_p = mean_train_loss_p / num_p
            else:
                mean_train_loss_u = 0.0
                mean_train_loss_v = 0.0
                mean_train_loss_p = 0.0


            train_relative_errors.append((mean_train_L2_u, mean_train_L2_v, mean_train_L2_p, mean_train_L2))
            train_loss.append((mean_train_loss_u, mean_train_loss_v, mean_train_loss_p))
                        
            print('Best L2 relative error:', err)
            print('Best u  relative error:', err_u)
            print('Best v  relative error:', err_v)
            print('Best p  relative error:', err_p)
            print('Best L2 train error:', mean_train_L2)  
            print('Best u  train error:', mean_train_L2_u)
            print('Best v  train error:', mean_train_L2_v)
            print('Best p  train error:', mean_train_L2_p)
            print('val_loss_u', val_loss_u)
            print('train_loss_u:', mean_train_loss_u)
            print('val_loss_v:', val_loss_v)
            print('train_loss_v', mean_train_loss_v)
            print('val_loss_p:', val_loss_p)
            print('train_loss_p:', mean_train_loss_p)


    # 定义文件路径变量
    file_path = 'relative_errors_{}.txt'.format(args.data)
        
    # 保存相对误差到文件
    if args.phase == 'train':
        if actual_epochs:
            save_relative_errors(train_loss, val_loss, train_relative_errors, val_relative_errors, actual_epochs, file_path, config, args)
            print("Relative errors have been saved to '{}'.".format(file_path))
            
    #  # 示例：读取并绘制相对误差图，文件路径是相对路径
    # plot_loss_from_file(file_path)
    # plot_relative_errors_from_file(file_path)
    # final test
    model.load_state_dict(torch.load(f'best_model_{args.data}.pth', map_location=device))   
    model.eval()

    err,err_u,err_v,err_p = test(model, test_loader, device, args)

    print('Best L2 relative error on test loader:', err)
    print('Best L2 relative error_u on test loader:', err_u)
    print('Best L2 relative error_v on test loader:', err_v)
    print('Best L2 relative error_p on test loader:', err_p)
    file_path = "relative_L2.txt"
    with open(file_path, "a") as f:
        f.write(f"TM {err_u} {err_v} {err_p}\n")
    
    

In [ ]:
with open(r'ns.yaml', 'r') as stream:
    config = yaml.load(stream, yaml.FullLoader)   
# 优化内存使用：减少采样点数
print("优化后的配置:", config)

In [ ]:
# 清理GPU内存
import torch
torch.cuda.empty_cache()
print(f"GPU内存已清理。当前GPU内存使用: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")

import argparse

args = argparse.Namespace(
    phase='train', 
    target_id='82', 
    data='ns', 
    model='DeepMONet', 

)

if args.model == 'DeepMONet':
    model = DeepMONet(config)
    
print("模型已创建。开始训练前GPU内存使用:", f"{torch.cuda.memory_allocated(0)/1024**3:.2f} GB")

In [ ]:
train(args, config, model, device, (train_loader, val_loader, test_loader))